# Task 2.1: bit-flip rate $\Gamma_X$

**Conventions.** $\hbar=1$, time in µs, rates in rad/µs. $\Gamma_X$ is the decay rate of $\mathrm{Re}\langle a\rangle$, equal to $-\lambda_1$ of the odd $Z_2$ block of the Liouvillian. Dubovitskii's rate is per direction, so it is $\Gamma_X/2$.

**Two channels.** Channel 1 is the code-space Golden Rule, $\Gamma^{(1)}=\kappa_a\alpha^2(\coth 2\alpha^2-1)$, so $\gamma=4$. Channel 2 leaves the code space and comes back, $\Gamma^{(2)}\approx c(\kappa_a^2/\kappa_2)e^{-2\alpha^2}$, so $\gamma=2$.

**Golden Rule (not graded).** $\langle-\alpha|\alpha\rangle=e^{-2\alpha^2}$, so $|\langle-\alpha|a|\alpha\rangle|^2=\alpha^2 e^{-4\alpha^2}$ and $\gamma_{\mathrm{FGR}}=4$. That is channel 1. It stops dominating once $\alpha^2\gtrsim 3$ (the naive crossover that drops the $\alpha^2$ prefactor is $2.3$): one photon loss leaves the code space, and confinement returns the state to the other well at a cost of only $e^{-2\alpha^2}$.

**Thermal photons.** $n_{\mathrm{th}}>0$ adds an $e^{-2\alpha^2}$ piece at first order in $\kappa_a$. The prefactor $c_{\mathrm{th}}$ is fitted. This does not promise the experimental $\gamma\approx\ln 4.2\approx 1.44$.


In [ ]:
import os, sys, json, subprocess
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))
from IPython.display import Image, display
show = lambda name, w=900: display(Image(f'figures/{name}.png', width=w))
RERUN = False


In [ ]:
from src.results import load_record
from tasks.task_2_1_channels import gamma_ch1
from src import config as C

if RERUN:
    for cmd in (
        [sys.executable, '-m', 'tasks.task_2_1_eig', '--quick'],
        [sys.executable, '-m', 'tasks.task_2_1_td', '--quick'],
        [sys.executable, '-m', 'tasks.task_2_1_channels', '--quick'],
        [sys.executable, '-m', 'tasks.task_2_1_thermal', '--quick'],
        [sys.executable, '-m', 'tasks.task_2_1_figures', '--quick'],
    ):
        subprocess.check_call(cmd)

def try_load(rid):
    try:
        return load_record(rid)
    except (FileNotFoundError, OSError, ValueError):
        return None

print('alpha  one_mode  two_mode  td_one  rel_td  ch1_share')
for a in (0.8, 1.0, 1.2, 1.4):
    one = try_load(f't2_1.gamma_x.eig.one_mode.a{a:.2f}.nth0')
    two = try_load(f't2_1.gamma_x.eig.two_mode.a{a:.2f}.nth0')
    td = try_load(f't2_1.gamma_x.td.one_mode.a{a:.2f}')
    if one is None:
        print(f'{a:.1f}  (no production records yet)')
        continue
    rel = abs(td['value'] - one['value']) / one['value'] if td else float('nan')
    share = gamma_ch1(a, C.KAPPA_A) / one['value']
    two_v = two['value'] if two else float('nan')
    td_v = td['value'] if td else float('nan')
    print(f'{a:.1f}  {one["value"]:.4e}  {two_v:.4e}  {td_v:.4e}  {rel:.3e}  {share:.3f}')


In [ ]:
for name in ('task_2_1_gamma_x', 'task_2_1_local_slope', 'task_2_1_kappa_a_sweep',
             'task_2_1_thermal', 'task_2_1_committor_wigner'):
    path = f'figures/{name}.png'
    if os.path.exists(path):
        show(name)
    else:
        print('missing', path)
